# EXP-2026-001 / Q4-O — leakage-free morphology baseline + current-beat raw-CNN residual

| | |
|---|---|
| Spec | `experiments/specs/EXP-2026-001-q4o-leakage-free-residual-cnn.md` |
| Module | `mit-bih/q4o_leakage_free_residual.py` (all evaluation / fold / OOF / statistics logic) |
| Tests | `mit-bih/test_q4o_leakage_free_residual.py` |
| Data | `/content/drive/MyDrive/mitbih/svdb_data5.npz` — the exact file Q4-N read |
| Runtime | **GPU** (Runtime -> Change runtime type -> GPU) |

This notebook is a **wrapper only**: Drive mount, config, run, visualise, save. It
deliberately contains no evaluation logic, no fold construction, no OOF stacking, and
no statistics — all of that lives in the module so it can be unit-tested.

## Why this run exists

Q4-N built the residual CNN's offset with a function that wrote **both** the train and
the test positions of one shared array, once per fold, in sequence:

```python
sc[tr] = lr.decision_function((X[tr] - mu) / sd)   # in-sample; the next fold overwrites it
sc[te] = lr.decision_function((X[te] - mu) / sd)
```

After the last of five folds, roughly **80%** of that array holds in-sample
predictions. So `cpu_comb = 0.8445`, `boost_fix = 0.8631`, and `boost_rank = 0.8492`
are **not** baselines and **not** improvements. They are carried here only as
contaminated reference values for the Arm E diagnostic.

## The five arms

| Arm | Name | Input | Offset |
|---|---|---|---|
| A | `morph_baseline` | frozen Q4-N morphology, 17 columns | — |
| B | `raw_current_cnn` | current beat, 2 leads, nothing else | — |
| **C** | `morph_plus_raw_residual` | current beat, 2 leads | cross-fitted morph logit |
| D | `shuffled_waveform_control` | current beat, **permuted within record** | cross-fitted morph logit |
| E | `corrected_q4n_diagnostic` | Q4-N 3-beat + 2 RR channels | cross-fitted `comb` logit |

**Primary**: `C − A`. **Key negative control**: `C − D`. Arm E is diagnostic only and
must not be read as a result or a new baseline.

## Pre-registered gates (all six required for PASS)

1. `mean(C − A) >= +0.015`
2. paired record-bootstrap 95% CI lower bound `> 0`
3. `mean(C − D) > 0` and its CI lower bound `> 0`
4. at least 4 of 5 seeds positive
5. lower-tail p10 of C not worse than A by more than `0.01`
6. every leakage / reproducibility assertion passes

**NO-GO does not mean "try a Transformer."** It means keep the morphology baseline and
go back to failure-record and lower-tail analysis. PASS does not mean Transformer
either — it means port the same minimal residual structure to MIT-BIH DS1→DS2.

## 1. Mount Drive and pull the repository code

`REPO_BRANCH` must be the branch carrying this experiment. The module is imported from
the checkout, never pasted into a cell — pasted code cannot be tested.

> **If you are re-running after a fix:** this cell purges the module from `sys.modules`
> before importing and then calls `Q.self_check()` in-process. A plain `import` is a
> no-op when the module is already loaded, so without this a `git pull` would leave the
> kernel executing the **old** code — the traceback would then show new source lines
> against old line numbers, and a fixed bug would look like it never went away. If the
> self-check fails, restart the runtime (Runtime → Restart session) and run again.

In [ ]:
import os, sys, subprocess, time, importlib

REPO_URL     = "https://github.com/ehdbddl06001-ui/my-github-test.git"
REPO_BRANCH  = "claude/exp-2026-001-q4o-leakage-free-residual"
REPO_DIR     = "/content/my-github-test"
NEED_VERSION = 4          # minimum q4o module version this notebook requires

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as exc:
    print("not Colab:", exc)
    DRIVE_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT", "/content")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    f"origin/{REPO_BRANCH}"], check=True)

MOD_DIR = os.path.join(REPO_DIR, "mit-bih")
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# ---------------------------------------------------------------------------
# Force a genuine re-import.
#
# `import q4o_leakage_free_residual` is a NO-OP once the module sits in
# sys.modules. Pulling new code updates the file on disk but NOT the code this
# kernel executes, and the test cell below runs in a subprocess -- so it reads
# the new file and passes while the kernel still runs the old one. That is how a
# fixed bug appears to persist. Purge the module and invalidate the import
# caches so the pull actually takes effect.
# ---------------------------------------------------------------------------
for _name in [m for m in sys.modules if m.startswith("q4o_leakage_free_residual")]:
    del sys.modules[_name]
importlib.invalidate_caches()

import q4o_leakage_free_residual as Q

# In-process proof that the fresh code is live -- runs the exact path that a
# stale import gets wrong (a cohort containing records below MIN_S/MIN_N).
check = Q.self_check(min_version=NEED_VERSION)

print("module     :", check["module_file"])
print("version    :", check["module_version"], "-", check["module_build"])
print("self-check : OK  (", check["n_scorable"], "of", check["n_record"],
      "records scorable,", check["n_unscored_beats"], "beats correctly unscored )")
print("drive root :", DRIVE_ROOT)
print("repo commit:", Q.git_commit_sha(REPO_DIR))
print("packages   :", Q.package_versions())
print("gpu        :", Q.gpu_info())

## 2. Run the tests first

If any test fails, stop. A failing leakage assertion invalidates the run before it
starts — that is criterion 6.

In [ ]:
rc = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, "mit-bih",
                                  "test_q4o_leakage_free_residual.py")],
    capture_output=True, text=True)
print(rc.stdout[-4000:])
if rc.returncode != 0:
    print(rc.stderr[-3000:])
    raise SystemExit("tests failed - do not run the experiment")

# The line above proves the FILE is good. It says nothing about the module this
# kernel has loaded, because it ran in a separate process. Re-assert in-kernel.
assert Q.MODULE_VERSION >= NEED_VERSION, (
    f"tests passed against the file, but this kernel holds q4o version "
    f"{Q.MODULE_VERSION}. Re-run the cell above, or restart the runtime.")
Q.self_check(min_version=NEED_VERSION)
print(f"tests passed · in-kernel module version {Q.MODULE_VERSION} verified")

## 3. Config

The data path is **fixed to `svdb_data5.npz`** — the exact file Q4-N read. Do not point
this at `svdb_data.npz`; it is a different, older file without `y3`/`sym`, and the
loader will refuse it. If the file is not where this cell expects it, stop and report
a blocker rather than substituting another file.

In [ ]:
# ---------------------------------------------------------------------------
# ANALYZE_EXISTING_RUN
#   True  -> no training at all. Reads an existing run bundle from EXISTING_RUN_DIR
#            and renders the full report. Nothing measured is recomputed or altered.
#   False -> run the experiment, then report on it.
# ---------------------------------------------------------------------------
ANALYZE_EXISTING_RUN = True
EXISTING_RUN_DIR     = "20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn"

DATA_PATH  = os.path.join(DRIVE_ROOT, "mitbih", "svdb_data5.npz")
PROJECT    = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
RUNS_DIR   = os.path.join(PROJECT, "runs")
REGISTRY   = os.path.join(PROJECT, "registry.jsonl")

SEEDS      = list(Q.TRAIN_SEEDS)   # five training seeds, pre-registered
EPOCHS     = Q.DL_EPOCH
BATCH      = Q.DL_BATCH
N_BOOT     = Q.NB_BOOT
PORT_CHECK = True                  # re-score Arm A under Q4-N's LORO (fidelity check)

if ANALYZE_EXISTING_RUN:
    OUT_DIR = (EXISTING_RUN_DIR if os.path.isabs(EXISTING_RUN_DIR)
               else os.path.join(RUNS_DIR, EXISTING_RUN_DIR))
    assert os.path.isdir(OUT_DIR), f"run bundle not found: {OUT_DIR}"
    print("MODE     : ANALYZE_EXISTING_RUN - no training, report only")
    print("run dir  :", OUT_DIR)
else:
    TIMESTAMP = time.strftime("%Y%m%dT%H%M", time.gmtime())
    OUT_DIR   = os.path.join(RUNS_DIR, Q.run_dir_name(TIMESTAMP))
    assert os.path.exists(DATA_PATH), (
        f"{DATA_PATH} not found. Do NOT substitute svdb_data.npz — stop and report a "
        f"blocker.")
    print("MODE     : FULL RUN")
    print("data     :", DATA_PATH)
    print("out dir  :", OUT_DIR)
    print("seeds    :", SEEDS)

print("k-sweep  :", Q.K_SWEEP, " operating points:", Q.K_OP)
print("gates    : gain >=", Q.GATE_MIN_GAIN, "| seeds >=", Q.GATE_MIN_SEED_AGREE,
      "| lower-tail drop <=", Q.GATE_LOWER_TAIL_MAX_DROP)

## 4. Load the cohort and record its provenance

Everything the manifest needs — absolute path, SHA256, shapes, dtypes, class and
record counts — is captured here, before any modelling.

In [ ]:
if ANALYZE_EXISTING_RUN:
    print("skipped - ANALYZE_EXISTING_RUN reads the finished bundle instead.")
    cohort = provenance = None
else:
    cohort, provenance = Q.load_cohort(DATA_PATH)

    print("file      :", provenance["file_name"])
    print("sha256    :", provenance["sha256"])
    print("beats     :", provenance["n_sample_labelled"], "of",
          provenance["n_sample_total"])
    print("shape     :", provenance["arrays"]["beat"]["shape"],
          provenance["arrays"]["beat"]["dtype"])
    print("classes   :", provenance["class_counts"])
    print("records   :", provenance["n_record"], "(record == patient:",
          provenance["record_equals_patient"], ")")

    rec_ok = Q.scorable_records(cohort)
    burden = Q.record_burden(cohort, rec_ok)
    fold_map = Q.make_fold_map(rec_ok, burden)
    Q.assert_fold_map_partition(fold_map, rec_ok, Q.N_OUTER_FOLDS)
    print("scorable  :", len(rec_ok), "records (MIN_S =", Q.MIN_S,
          ", MIN_N =", Q.MIN_N, ")")
    for f in range(Q.N_OUTER_FOLDS):
        rs = sorted(r for r in rec_ok if fold_map[r] == f)
        print(f"  fold {f}: {len(rs):2d} records  {rs}")

## 5. Run

Every arm, every seed, every fold. The runner raises on any leakage violation rather
than reporting a number, so reaching the end is itself part of criterion 6.

This is 5 seeds × 4 neural arms × 5 folds = **100 model trainings**. Budget
**1–3 hours** on a Colab GPU and keep the tab alive; the per-batch bottleneck is
CPU-side fancy indexing of the waveform array, not the GPU.

Peak host memory is roughly 3–4 GB (Arm E's 3-beat input is ~1.8 GB on its own), which
fits a standard Colab runtime but leaves little headroom — restart the runtime before
running if you have already loaded other large arrays.

Note that 22 of SVDB's 78 records fall below `MIN_S`/`MIN_N` and are not scored. Their
beats stay in the cohort, are absent from the fold map, and appear as `NaN` in
`probs.npy` with `fold = -1` and `scored_mask = False` in `predictions.npz`. That is
by design, not a failure.

In [ ]:
if ANALYZE_EXISTING_RUN:
    print("skipped - no training in ANALYZE_EXISTING_RUN mode.")
    import json
    result = json.load(open(os.path.join(OUT_DIR, "result.json"), encoding="utf-8"))
    print("loaded the existing result.json · verdict:", result["gates"]["verdict"])
else:
    log = Q.RunLog()
    result = Q.run_experiment(
        cohort, provenance, OUT_DIR,
        seeds=SEEDS, epochs=EPOCHS, batch=BATCH, n_boot=N_BOOT,
        port_check=PORT_CHECK, smoke=False, log=log)
    print("\nverdict:", result["gates"]["verdict"])

In [ ]:
if ANALYZE_EXISTING_RUN:
    print("skipped - the registry already carries this run; reporting never re-appends.")
else:
    record = {
        "run_id": Q.run_dir_name(TIMESTAMP),
        "experiment_id": Q.EXPERIMENT_ID,
        "arm_id": Q.ARM_ID,
        "primary_metric": result["primary_metric"],
        "primary_value": result["contrasts"]["C_minus_A"]["record_bootstrap"]["mean"],
        "primary_ci": [result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_low"],
                       result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_high"]],
        "negative_control": result["contrasts"]["C_minus_D"]["record_bootstrap"]["mean"],
        "verdict": result["gates"]["verdict"],
        "conclusion": (
            f"C-A {result['contrasts']['C_minus_A']['record_bootstrap']['mean']:+.4f}, "
            f"C-D {result['contrasts']['C_minus_D']['record_bootstrap']['mean']:+.4f}, "
            f"{result['gates']['verdict']}"),
        "run_folder": OUT_DIR,
        "data_sha256": provenance["sha256"],
        "git_commit": Q.git_commit_sha(REPO_DIR),
    }
    Q.append_registry(REGISTRY, record)
    print(json.dumps(record, indent=2))

## 6. 보고서 생성 (presentation only)

`Q.generate_report()` 는 완성된 run 번들을 **읽기만** 한다. 학습하지 않고, arm·fold·
seed·지표·bootstrap·gate 를 건드리지 않으며, `result.json` / `manifest.json` /
`config.json` / `fold_map.json` / `predictions.npz` / `arms/*/probs.npy` 에 쓰지 않는다.

두 가지를 스스로 검증한다.

1. **재현** — 저장된 logit 에서 per-record k-sweep 을 다시 계산해 `result.json` 의
   요약값과 일치하는지 확인한다. 어긋나면 보고서를 만들지 않고 예외를 던진다.
2. **불변** — 위 파일들의 SHA256 을 보고서 생성 전후로 비교한다. 하나라도 바뀌면 예외.

In [ ]:
report = Q.generate_report(OUT_DIR)

print()
print("재현 확인 :", report["reconciliation"])
print("측정 산출물 불변 :", report["fingerprint_stable"])
print("training history :",
      "있음" if report["training_history_present"] else "없음 (이 run 은 기록 기능 이전)")

## 7. Executive Summary

사람이 가장 먼저 읽어야 할 부분. 모든 수치는 `result.json` 에서 그대로 가져온다.

In [ ]:
print(report["executive_summary_ko"])

## 8. 표와 그림

그림의 축 라벨은 영어다 — Colab 에 기본 CJK 폰트가 없어 한글이 두부(□)로 깨진다.
해석은 각 그림 아래에 한국어로 붙인다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown
import pandas as pd

FIG = os.path.join(OUT_DIR, "figures")

def show(name, caption):
    """Render one report figure with its Korean interpretation underneath."""
    path = os.path.join(FIG, name)
    if not os.path.exists(path):
        display(Markdown(f"*(`{name}` 없음 — 이 run 에서는 생성되지 않았다)*"))
        return
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(14, 14 * h / w))
    ax.imshow(img); ax.axis("off")
    plt.show()
    display(Markdown(caption))

bundle = Q.load_run_bundle(OUT_DIR)
res    = bundle.result
ca     = res["contrasts"]["C_minus_A"]["record_bootstrap"]
cd     = res["contrasts"]["C_minus_D"]["record_bootstrap"]
a_ksw  = res["arms"][Q.ARM_A]["seed_averaged_ksw"]
c_ksw  = res["arms"][Q.ARM_C]["seed_averaged_ksw"]
stall  = Q.training_stalled(bundle, Q.ARM_C)
n_seed = len(bundle.seeds)

In [ ]:
display(Markdown("### 1. arm 요약표"))
display(pd.read_csv(os.path.join(FIG, "arm_metrics.csv")))
show("arm_summary_table.png",
     f"**해석.** morphology baseline(A) 의 k-sweep 달성률은 **{a_ksw['mean']:.4f}**, "
     f"주 비교군 C 는 **{c_ksw['mean']:.4f}** 다. "
     f"`Δ vs A` 열은 같은 record·같은 seed 로 짝지은 대비이며, 짝지은 대비가 없는 "
     f"arm 만 평균의 차이(\\*)로 표기했다. "
     f"seed SD 열은 seed 간 변동으로, 이 값보다 작은 차이는 해석하지 않는다.")

In [ ]:
display(Markdown("### 2. 주 비교 (확대)"))
show("primary_contrasts_zoom.png",
     f"**해석.** C−A = **{ca['mean']:+.4f}** [{ca['ci_low']:+.4f}, {ca['ci_high']:+.4f}], "
     f"C−D = **{cd['mean']:+.4f}** [{cd['ci_low']:+.4f}, {cd['ci_high']:+.4f}] 이다. "
     f"통과 기준은 평균 ≥ +{Q.GATE_MIN_GAIN} 이고 CI 하한 > 0 이므로, 점과 오차막대가 "
     f"파란 점선에 닿지 못하거나 CI 가 0(굵은 세로선)을 포함하면 실패다. "
     f"C−D 는 음성 대조로, 이 값이 0 이면 C 가 얻은 것은 박동 단위 파형 정보가 아니다.")

In [ ]:
display(Markdown("### 3. 참조용 큰 차이 (축 분리)"))
show("reference_gap_separate.png",
     "**해석.** B−A(원파형 CNN 단독 − 형태 baseline)처럼 규모가 다른 대비는 "
     "여기 따로 그린다. 이전 `contrasts.png` 는 이 값과 주 비교를 같은 축에 그려서 "
     "±0.001 규모의 주 비교가 모두 0 근처의 한 점으로 뭉개졌다. "
     "축을 나눈 것은 표현의 문제일 뿐, 어떤 측정값도 바뀌지 않았다.")

In [ ]:
display(Markdown("### 4. k 별 달성률"))
show("achievement_by_k.png",
     f"**해석.** 왼쪽은 각 arm 의 achievement@k 이고, 오른쪽은 A 대비 C 의 차이를 "
     f"k 마다 확대한 것이다. 문헌 판독 운영점(k=30~50)에서의 거동이 임상적으로 가장 "
     f"중요하며, 형태 축의 이득은 k 가 작을수록 커지는 것이 지금까지의 패턴이었다. "
     f"오른쪽 패널은 seed 평균 간의 차이이므로 짝지은 신뢰구간이 아니다 — 판정은 "
     f"2번 그림의 CI 로만 한다.")

In [ ]:
display(Markdown("### 5. seed 별 방향"))
show("seed_effects.png",
     f"**해석.** {n_seed}개 seed 각각의 C−A 와 C−D 다. gate 4 는 "
     f"{Q.GATE_MIN_SEED_AGREE}/{n_seed} 이상의 seed 가 같은 양(+) 방향일 것을 요구하며, "
     f"이번 run 은 **{res['contrasts']['C_minus_A']['positive_seed_count']}/{n_seed}** 이다. "
     f"방향이 seed 마다 뒤집히면 평균이 양수여도 신뢰할 수 없다는 뜻이다.")

In [ ]:
display(Markdown("### 6. fold·seed 학습 진단"))
_warn = ("\n\n> ⚠️ **이 run 에서 Arm C 는 "
         f"{stall.get('n_best_epoch_zero')}/{stall.get('n_total')} (seed × fold) 전부에서 "
         "첫 번째 학습 epoch 완료 후의 체크포인트(`best_epoch = 0`)가 선택됐다.** "
         "이후 epoch 는 dev BCE 를 개선하지 못했다는 뜻이다. epoch 0 은 학습 전 상태가 "
         "아니다 - 한 epoch 분량(약 77~79 optimizer step)의 업데이트를 거쳤고, 선택된 "
         "체크포인트의 alpha 도 0 이 아니다(대체로 |0.078~0.101|). 이 run 은 학습 전 "
         "체크포인트(epoch -1)를 dev 후보로 평가하지 않았으므로, epoch 0 이 정확한 "
         "morphology baseline 보다 개선됐는지는 판정할 수 없다. alpha 부호는 head 부호와 "
         "함께 뒤집힐 수 있으므로 부호 자체를 seed 불안정성으로 읽지 말 것 - 해석 대상은 "
         "alpha × residual 출력이다.") if stall.get("all_zero") else ""
show("fold_training_diagnostics.png",
     "**해석.** 행은 seed, 열은 fold 다. `alpha` 는 학습된 잔차 스케일(0 에서 출발), "
     "`best_epoch` 는 early stopping 이 고른 epoch, `dev_loss` 는 그 지점의 dev BCE 손실이다. "
     "alpha 가 0 근처에 머물면 잔차가 사실상 꺼져 있다는 뜻이다." + _warn)

In [ ]:
display(Markdown("### 7. 레코드(환자)별 delta"))
show("patient_delta_waterfall.png",
     f"**해석.** seed {n_seed}개를 **모두 평균한** record 별 C−A 를 정렬한 것이다"
     f"(한 seed 만 쓴 그림이 아니다). 파란색은 개선, 빨간색은 악화다. "
     f"평균이 0 이어도 개선과 악화가 서로 상쇄된 것인지, 아무 record 도 움직이지 "
     f"않은 것인지는 전혀 다른 이야기이며, 이 그림이 그것을 구분해 준다.")

pdf_ = pd.read_csv(os.path.join(FIG, "patient_delta.csv"))
display(Markdown("**개선 상위 10 record**"))
display(pdf_.nlargest(10, "delta_C_minus_A"))
display(Markdown("**악화 상위 10 record**"))
display(pdf_.nsmallest(10, "delta_C_minus_A"))
display(Markdown(
    f"**해석.** `s_burden` 은 그 record 의 S 비율, `ksw_A`/`ksw_C` 는 각 arm 의 성능이다. "
    f"개선·악화가 특정 burden 구간이나 baseline 성능 구간에 몰려 있는지 보라 — "
    f"몰려 있다면 다음 실험은 그 하위집단을 표적으로 삼아야 한다. "
    f"전체 {len(pdf_)}개 record 중 {int((pdf_['delta_C_minus_A'] > 0).sum())}개가 개선됐다."))

In [ ]:
display(Markdown("### 8. 환자별 분포"))
show("metric_distribution.png",
     f"**해석.** A/C/D 의 record 별 k-sweep 분포다. 평균 하나로는 보이지 않는 "
     f"하위꼬리를 보기 위한 그림이며, 빨간 선이 p10(하위 10 백분위), 초록 선이 중앙값이다. "
     f"gate 5 는 C 의 p10 이 A 대비 {Q.GATE_LOWER_TAIL_MAX_DROP} 이상 나빠지지 않을 것을 "
     f"요구한다 — 평균이 올라도 최약 환자가 나빠지면 임상적으로 쓸 수 없기 때문이다.")

In [ ]:
display(Markdown("### 9. 학습 곡선"))
if report["training_history_present"]:
    show("learning_curves.png",
         "**해석.** (seed, fold) 마다 한 선이다. epoch 별 train/dev BCE 손실, dev PR-AUC, "
         "alpha 를 보여 준다. 체크포인트 선택은 **dev BCE 손실만** 사용하며, dev PR-AUC 는 "
         "기록 전용이라 선택에 개입하지 않는다.")
else:
    display(Markdown(
        "**이 run 에는 epoch 단위 training history 가 없다.** 기록 기능은 이번 개정에서 "
        "추가됐으므로 *이후* 실행부터 `training_history.json` 과 `learning_curves.png` 가 "
        "생성된다. 없는 데이터를 지어내지 않았고, 학습 곡선도 그리지 않았다."))

## 9. report_summary.md

모든 핵심 수치, PASS/FAIL 근거, baseline 정의, Q4-N `0.8631` 을 제외한 이유, 구조 설명,
한계와 다음 결정, 생성한 그림 링크가 한 문서에 들어 있다.

In [ ]:
display(Markdown(open(report["report_markdown"], encoding="utf-8").read()))

## 9. Bring the run back into GitHub

1. Save this executed notebook to `notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb`
   and commit it — an unexecuted notebook is not evidence.
2. Ingest the measured result:

```bash
python pipelines/ingest_run.py \
    --results <run_dir>/result.json \
    --notebook notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb
```

3. Register the run bundle in `research/ASSETS.md` (path, not a move).
4. Open the review PR with the exact commands and any deviations.

Do **not** update `research/PROJECT_STATE.md` with a new baseline until the design
owner has read the executed notebook and the measured result. Until then this
experiment has no outcome.

In [ ]:
print("run bundle:", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    for f_ in sorted(files):
        p = os.path.join(root, f_)
        print(f"  {os.path.relpath(p, OUT_DIR):<48} {os.path.getsize(p):>10,d} bytes")
Q.verify_bundle(OUT_DIR)
print("\nbundle schema verified")